In [1]:
import pandas as pd
import numpy as np
import os
import joblib

print("Unified prediction pipeline environment ready!")

Unified prediction pipeline environment ready!


In [2]:
data_path = "../data/CICIDS2017/ml_ready/cicids2017_binary_ml_ready.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (288277, 79)


In [3]:
rf_model_path = "../models/random_forest_baseline.joblib"
isolation_model_path = "../models/isolation_forest_anomaly.joblib"

rf_model = joblib.load(rf_model_path)
isolation_model = joblib.load(isolation_model_path)

print("Random Forest loaded successfully!")
print("Isolation Forest loaded successfully!")

Random Forest loaded successfully!
Isolation Forest loaded successfully!


In [4]:
columns_to_drop = [
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Timestamp",
    "Label",
    "Binary_Label"
]

feature_columns = [
    column for column in df.columns
    if column not in columns_to_drop
]

print("Number of model features:", len(feature_columns))
print("\nFirst 10 features:")
print(feature_columns[:10])

Number of model features: 78

First 10 features:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']


In [5]:
X = df[feature_columns].copy()

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

valid_rows = X.notna().all(axis=1)

X = X.loc[valid_rows].copy()

print("Valid feature matrix shape:", X.shape)

Valid feature matrix shape: (288277, 78)


In [6]:
print("Random Forest expected features:",
      rf_model.n_features_in_)

print("Isolation Forest expected features:",
      isolation_model.n_features_in_)

print("Pipeline features:",
      X.shape[1])

if (
    rf_model.n_features_in_ == X.shape[1]
    and isolation_model.n_features_in_ == X.shape[1]
):
    print("\nModel compatibility check: PASSED")
else:
    print("\nModel compatibility check: FAILED")

Random Forest expected features: 78
Isolation Forest expected features: 78
Pipeline features: 78

Model compatibility check: PASSED


In [7]:
sample_index = 0

sample_flow = X.iloc[[sample_index]]

print("Sample network flow selected!")
print("Shape:", sample_flow.shape)

Sample network flow selected!
Shape: (1, 78)


In [8]:
attack_probability = rf_model.predict_proba(
    sample_flow
)[:, 1][0]

attack_prediction = rf_model.predict(
    sample_flow
)[0]

print("Attack probability:",
      round(attack_probability * 100, 2), "%")

print("Random Forest prediction:",
      "ATTACK" if attack_prediction == 1 else "BENIGN")

Attack probability: 0.0 %
Random Forest prediction: BENIGN


In [9]:
isolation_prediction = isolation_model.predict(
    sample_flow
)[0]

anomaly_flag = 1 if isolation_prediction == -1 else 0

print("Anomaly status:",
      "ANOMALY" if anomaly_flag == 1 else "NORMAL")

Anomaly status: NORMAL


In [10]:
attack_score = attack_probability * 100

risk_score = (
    0.70 * attack_score +
    0.30 * (anomaly_flag * 100)
)

risk_score = float(
    np.clip(risk_score, 0, 100)
)

print("Attack score:",
      round(attack_score, 2))

print("Risk score:",
      round(risk_score, 2))

Attack score: 0.0
Risk score: 0.0


In [11]:
def get_risk_level(score):
    if score < 25:
        return "LOW"
    elif score < 50:
        return "MEDIUM"
    elif score < 75:
        return "HIGH"
    else:
        return "CRITICAL"


risk_level = get_risk_level(risk_score)

print("Risk level:", risk_level)

Risk level: LOW


In [12]:
prediction_result = {
    "attack_probability": round(
        attack_probability * 100, 2
    ),
    "prediction": (
        "ATTACK"
        if attack_prediction == 1
        else "BENIGN"
    ),
    "anomaly": (
        "YES"
        if anomaly_flag == 1
        else "NO"
    ),
    "risk_score": round(
        risk_score,
        2
    ),
    "risk_level": risk_level
}

print("Unified prediction result:")
print(prediction_result)

Unified prediction result:
{'attack_probability': np.float64(0.0), 'prediction': 'BENIGN', 'anomaly': 'NO', 'risk_score': 0.0, 'risk_level': 'LOW'}


In [13]:
prediction_df = pd.DataFrame(
    [prediction_result]
)

print(prediction_df)

   attack_probability prediction anomaly  risk_score risk_level
0                 0.0     BENIGN      NO         0.0        LOW


In [14]:
sample_size = 10

sample_flows = X.head(sample_size)

rf_probabilities = rf_model.predict_proba(
    sample_flows
)[:, 1]

rf_predictions = rf_model.predict(
    sample_flows
)

isolation_predictions = isolation_model.predict(
    sample_flows
)

anomaly_flags = np.where(
    isolation_predictions == -1,
    1,
    0
)

risk_scores = (
    0.70 * (rf_probabilities * 100)
    +
    0.30 * (anomaly_flags * 100)
)

risk_scores = np.clip(
    risk_scores,
    0,
    100
)

risk_levels = [
    get_risk_level(score)
    for score in risk_scores
]

print("Multiple-flow prediction completed!")

Multiple-flow prediction completed!


In [15]:
batch_results = pd.DataFrame({
    "Attack_Probability": rf_probabilities * 100,
    "Prediction": np.where(
        rf_predictions == 1,
        "ATTACK",
        "BENIGN"
    ),
    "Anomaly": np.where(
        anomaly_flags == 1,
        "YES",
        "NO"
    ),
    "Risk_Score": risk_scores,
    "Risk_Level": risk_levels
})

batch_results["Attack_Probability"] = (
    batch_results["Attack_Probability"].round(2)
)

batch_results["Risk_Score"] = (
    batch_results["Risk_Score"].round(2)
)

print(batch_results)

   Attack_Probability Prediction Anomaly  Risk_Score Risk_Level
0                 0.0     BENIGN      NO         0.0        LOW
1                 0.0     BENIGN      NO         0.0        LOW
2                 0.0     BENIGN     YES        30.0     MEDIUM
3                 0.0     BENIGN     YES        30.0     MEDIUM
4                 0.0     BENIGN     YES        30.0     MEDIUM
5                 0.0     BENIGN      NO         0.0        LOW
6                 0.0     BENIGN      NO         0.0        LOW
7                 0.0     BENIGN      NO         0.0        LOW
8                 0.0     BENIGN      NO         0.0        LOW
9                 1.0     BENIGN      NO         0.7        LOW


In [16]:
output_dir = "../data/CICIDS2017/ml_ready"

os.makedirs(
    output_dir,
    exist_ok=True
)

prediction_output = os.path.join(
    output_dir,
    "cicids2017_prediction_pipeline_results.csv"
)

batch_results.to_csv(
    prediction_output,
    index=False
)

print("Prediction results saved successfully!")
print("Path:", prediction_output)

Prediction results saved successfully!
Path: ../data/CICIDS2017/ml_ready\cicids2017_prediction_pipeline_results.csv


In [17]:
def predict_network_flow(flow_data):
    """
    Predict network security risk for a single network flow.
    """

    if isinstance(flow_data, pd.Series):
        flow_data = flow_data.to_frame().T

    flow = flow_data.copy()

    flow = flow.drop(
        columns=[
            "Flow ID",
            "Source IP",
            "Destination IP",
            "Timestamp",
            "Label",
            "Binary_Label"
        ],
        errors="ignore"
    )

    flow = flow.reindex(
        columns=feature_columns,
        fill_value=0
    )

    flow = flow.replace(
        [np.inf, -np.inf],
        np.nan
    )

    flow = flow.fillna(0)

    attack_probability = rf_model.predict_proba(
        flow
    )[:, 1][0]

    prediction = rf_model.predict(
        flow
    )[0]

    isolation_prediction = isolation_model.predict(
        flow
    )[0]

    anomaly_flag = (
        1 if isolation_prediction == -1
        else 0
    )

    attack_score = attack_probability * 100

    risk_score = (
        0.70 * attack_score +
        0.30 * (anomaly_flag * 100)
    )

    risk_score = float(
        np.clip(risk_score, 0, 100)
    )

    risk_level = get_risk_level(
        risk_score
    )

    return {
        "prediction": (
            "ATTACK"
            if prediction == 1
            else "BENIGN"
        ),
        "attack_probability": round(
            attack_probability * 100,
            2
        ),
        "anomaly": (
            "YES"
            if anomaly_flag == 1
            else "NO"
        ),
        "risk_score": round(
            risk_score,
            2
        ),
        "risk_level": risk_level
    }


print("Reusable prediction function created successfully!")

Reusable prediction function created successfully!


In [18]:
test_flow = df.loc[
    valid_rows
].iloc[0]

result = predict_network_flow(
    test_flow
)

print("Prediction pipeline test:")
print(result)

Prediction pipeline test:
{'prediction': 'BENIGN', 'attack_probability': np.float64(0.0), 'anomaly': 'NO', 'risk_score': 0.0, 'risk_level': 'LOW'}


In [19]:
print("=" * 65)
print("UNIFIED NETWORK SECURITY PREDICTION PIPELINE")
print("=" * 65)

print("Dataset                : CICIDS2017")
print("Supervised Model       : Random Forest")
print("Anomaly Model          : Isolation Forest")
print("Risk Scoring           : 70% Attack Probability + 30% Anomaly")
print("Risk Range             : 0 - 100")
print()
print("Pipeline:")
print("Network Flow")
print("     ↓")
print("Feature Preparation")
print("     ↓")
print("Random Forest")
print("     ↓")
print("Attack Probability")
print("     +")
print("Isolation Forest")
print("     ↓")
print("Anomaly Detection")
print("     ↓")
print("Risk Scoring")
print("     ↓")
print("Risk Level")
print("     ↓")
print("Security Prediction")
print("=" * 65)

print("\nPipeline verification completed successfully!")

UNIFIED NETWORK SECURITY PREDICTION PIPELINE
Dataset                : CICIDS2017
Supervised Model       : Random Forest
Anomaly Model          : Isolation Forest
Risk Scoring           : 70% Attack Probability + 30% Anomaly
Risk Range             : 0 - 100

Pipeline:
Network Flow
     ↓
Feature Preparation
     ↓
Random Forest
     ↓
Attack Probability
     +
Isolation Forest
     ↓
Anomaly Detection
     ↓
Risk Scoring
     ↓
Risk Level
     ↓
Security Prediction

Pipeline verification completed successfully!
